Vogliamo costruire un dataset Amazon-like multi-categoria, intersecando:
Amazon Reviews 2023 che contiene metadati + rating/recensioni
Amazon question/answer storico McAuley

Questo notebook costruisce il dataset test_dataset2 intersecando più categorie:
- QA storico McAuley
- metadati Amazon Reviews 2023
- rating/recensioni Amazon Reviews 2023

Categorie usate:
- Electronics
- Cell Phones and Accessories
- Video Games
- Software
- Office Products
- Appliances

Output finale:
- items.parquet
- qa.parquet
- ratings.parquet
- category_summary.csv
- summary.json

In [1]:
# Se mancano pacchetti, esegui questa cella una sola volta.
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: C:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path #per gestire meglio percorsi di file e cartelle
import ast #servirà più avanti per leggere file del QA storico
import gzip #leggere e scrivere file compressi gz
import json #lavorare con json
from typing import Iterable, Optional #tipi
import pandas as pd #pandas per creare dataframe
import pyarrow as pa #per definire schemi tabellari
import pyarrow.parquet as pq    #per leggere metadata dei file .parquet
import requests #per scaricare da URL
from tqdm.auto import tqdm #barra di progresso
import datasets #pacchetto datasets
from datasets import load_dataset #per caricare dataset

datasets.logging.set_verbosity_error() #riduce i messaggi stampati da datasets

C:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#configurazione
DATASET_NAME="test_dataset2" #nome tecnico usato per cartelle di output
DATASET_LABEL="AmazonLike_v1" #nome leggibile del dataset

CATEGORIES=[ #categorie scelte per avere nucleo tech e un po' di varietà
    {"label":"Electronics","hf_category":"Electronics","qa_slug":"Electronics","group":"tech"},
    {"label":"Cell Phones and Accessories","hf_category":"Cell_Phones_and_Accessories","qa_slug":"Cell_Phones_and_Accessories","group":"tech"},
    {"label":"Video Games","hf_category":"Video_Games","qa_slug":"Video_Games","group":"tech"},
    {"label":"Software","hf_category":"Software","qa_slug":"Software","group":"tech"},
    {"label":"Office Products","hf_category":"Office_Products","qa_slug":"Office_Products","group":"office"},
    {"label":"Appliances","hf_category":"Appliances","qa_slug":"Appliances","group":"home"},
]
#label è il nome della categoria che usiamo noi
#hfcategory è il nome usato su amazon reviews
#qaslug è il nome del file qa di quella categoria
#group macro categoria creata da me

QA_BASE_URL="https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa"
MAX_REVIEWS_SCAN_BY_CATEGORY=None #None significa scansiona tutte le review disponibili
SAVE_TEXT_OFREVIEW=True
FORCE_REBUILD_RATINGS=False #se True ricrea ratings.parquet anche se esiste già

In [4]:
PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
RAW_DIR=PROJECT_ROOT/"data"/"raw"
PROCESSED_DIR=PROJECT_ROOT/"data"/"processed"/f"{DATASET_NAME}_intersection"
QA_RAW_DIR=RAW_DIR/"qa"

ITEMS_OUT=PROCESSED_DIR/"items.parquet" #percorso finale dei prodotti
QA_OUT=PROCESSED_DIR/"qa.parquet" #percorso file domande/risposte
RATINGS_OUT=PROCESSED_DIR/"ratings.parquet" #percorso rating
CATEGORY_SUMMARY_PATH=PROCESSED_DIR/"category_summary.csv" #riepilogo per categoria
SUMMARY_PATH=PROCESSED_DIR/"summary.json" #riepilogo generale

RAW_DIR.mkdir(parents=True,exist_ok=True) #crea data/raw
QA_RAW_DIR.mkdir(parents=True,exist_ok=True) #crea data/raw/qa
PROCESSED_DIR.mkdir(parents=True,exist_ok=True) #crea cartella output dataset2

print("Project root:",PROJECT_ROOT)
print("Output:",PROCESSED_DIR)
print("Categorie selezionate:")
for category in CATEGORIES:
    print("-",category["label"],"| hf=",category["hf_category"],"| qa=",category["qa_slug"])


Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Output: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection
Categorie selezionate:
- Electronics | hf= Electronics | qa= Electronics
- Cell Phones and Accessories | hf= Cell_Phones_and_Accessories | qa= Cell_Phones_and_Accessories
- Video Games | hf= Video_Games | qa= Video_Games
- Software | hf= Software | qa= Software
- Office Products | hf= Office_Products | qa= Office_Products
- Appliances | hf= Appliances | qa= Appliances


In [5]:
def download_file(url:str,output_path:Path,force:bool=False)->Path: #scarica da url in output_path
    if output_path.exists() and not force: #non riscarica
        print(f"Già presente: {output_path}")
        return output_path
    output_path.parent.mkdir(parents=True,exist_ok=True) #se non esiste cartella di destinazione la crea

    with requests.get(url,stream=True,timeout=60)as r:
        r.raise_for_status()
        total=int(r.headers.get("content-length",0)) #dimensione
        with open(output_path,"wb") as f,tqdm( #apre file in scrittura binaria, perchè scarichi .gz e apre barra tqdm
            total=total,
            unit="B", #in byte
            unit_scale=True,
            desc=output_path.name,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return output_path

In [6]:
#Convertiamo QA storico da JSONL gzip a JSON e aggiungiamo parent_asin per
#agganciarlo al dataset amazon 2023
#per fare questo l'asin storico lo usiamo come parent_asin

def iter_historical_qa(path:Path)->Iterable[dict]: #legge qa storico, restituisce dizionari python
    with gzip.open(path,"rt",encoding="utf-8",errors="replace") as f: #apre il file .gz
        for line in f:
            line=line.strip() #rimuove spazi e newline
            if line:
                yield ast.literal_eval(line) #converte riga da testo a dizionario python

#dizionari per info per ogni categoria
qa_info_by_category={}
category_summary_rows=[]

for category in CATEGORIES: #cicliamo ogni categoria
    qa_slug=category["qa_slug"]
    qa_url=f"{QA_BASE_URL}/qa_{qa_slug}.json.gz"
    #file originale scaricato e file convertito
    qa_raw_path=QA_RAW_DIR/f"qa_{qa_slug}.json.gz"
    qa_strict_path=QA_RAW_DIR/f"qa_{qa_slug}.strict.jsonl.gz"

    download_file(qa_url,qa_raw_path) #scarica da internet

    qa_asins=set()
    qa_rows=0

    #Riusa i QA convertiti solo se coerenti con la categoria
    if qa_strict_path.is_file():
        with gzip.open(qa_strict_path,"rt",encoding="utf-8") as existing:
            for qa_id,line in enumerate(existing):
                row=json.loads(line)
                expected={"source_category":category["label"],"hf_category":category["hf_category"],"qa_slug":qa_slug,"category_group":category["group"]}
                if any(row.get(key)!=value for key,value in expected.items()) or row.get("qa_id")!=qa_id or row.get("parent_asin")!=row.get("asin"):
                    raise ValueError(f"QA convertiti incoerenti: {qa_strict_path}")
                if row.get("parent_asin"):
                    qa_asins.add(row["parent_asin"])
                qa_rows+=1
    else:
        with gzip.open(qa_strict_path,"wt",encoding="utf-8") as out: #apre il file in scrittura, .jsonl.gz
            for qa_id,row in enumerate(tqdm(iter_historical_qa(qa_raw_path),desc=f"Conversione QA {category['label']}")):
                asin=row.get("asin") #prende valore dal campo asin
                row["qa_id"]=qa_id #aggiunge id numerico alla riga qa nella categoria
                row["parent_asin"]=asin #crea il campo parent_asin e ci copia l'asin
                row["source_category"]=category["label"] #categoria
                row["hf_category"]=category["hf_category"] #categoria huggingface
                row["qa_slug"]=category["qa_slug"] #file qa da cui arriva
                row["category_group"]=category["group"] #macrogruppo
    
                if asin:
                    qa_asins.add(asin)
    
                out.write(json.dumps(row,ensure_ascii=False)+"\n") #scrive la riga nel nuovo file jsonl
                qa_rows+=1

    qa_info_by_category[category["label"]]={ #info qa per categoria
        "qa_raw_path":qa_raw_path,
        "qa_strict_path":qa_strict_path,
        "qa_asins":qa_asins,
        "qa_rows":qa_rows,
        "qa_products":len(qa_asins),
    }
    category_summary_rows.append({ #riepilogo senza asin, più leggero
        "source_category":category["label"],
        "hf_category":category["hf_category"],
        "qa_slug":category["qa_slug"],
        "category_group":category["group"],
        "qa_rows":qa_rows,
        "qa_products":len(qa_asins),
    })

    print(category["label"],"| QA rows:",qa_rows,"| prodotti QA:",len(qa_asins),"| file strict:",qa_strict_path)

Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.json.gz


Electronics | QA rows: 314263 | prodotti QA: 39371 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.strict.jsonl.gz
Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Cell_Phones_and_Accessories.json.gz


Cell Phones and Accessories | QA rows: 85865 | prodotti QA: 10407 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Cell_Phones_and_Accessories.strict.jsonl.gz
Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Video_Games.json.gz
Video Games | QA rows: 13307 | prodotti QA: 1188 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Video_Games.strict.jsonl.gz
Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Software.json.gz
Software | QA rows: 10636 | prodotti QA: 1191 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Software.strict.jsonl.gz
Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Office_Products.json.gz


Office Products | QA rows: 43608 | prodotti QA: 5681 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Office_Products.strict.jsonl.gz
Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Appliances.json.gz
Appliances | QA rows: 9011 | prodotti QA: 1153 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Appliances.strict.jsonl.gz


In [7]:
#prepariamo funzioni comuni e schema dei prodotti
#ogni riga salvata mantiene anche source_category per analisi multi-categoria

def image_url(images)->Optional[str]: #prova a estrarre immagine da campo images
    if isinstance(images,dict):
        for key in ("hi_res","large","thumb"):
            values=images.get(key) or []
            for value in values:
                if value:
                    return value
    if isinstance(images,list):
        for img in images:
            if isinstance(img,dict):
                for key in ("large_image_url","medium_image_url","small_image_url"):
                    value=img.get(key)
                    if value:
                        return value
    return None

def tojson_string(value)->str: #converte valore in stringa json
    if value is None:
        return "null"
    return json.dumps(value,ensure_ascii=False)

def tofloat_or_none(value): #converte valore in numero decimale
    try:
        if value in (None,"","None"):
            return None
        return float(value)
    except Exception:
        return None

def toint_or_none(value): #converte in intero
    try:
        if value in (None,"","None"):
            return None
        return int(value)
    except Exception:
        return None

def get_category(row:dict,default_category:str)->Optional[str]:
    main_category=row.get("main_category")
    if main_category not in (None,"","None"):
        return main_category

    categories=row.get("categories")
    if isinstance(categories,list) and len(categories)>0:
        return categories[0]

    return default_category

ITEM_SCHEMA=pa.schema([ #schema del file items.parquet
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("title",pa.string()),
    ("category",pa.string()),
    ("avg_rating",pa.float64()),
    ("num_ratings",pa.int64()),
    ("price",pa.float64()),
    ("store",pa.string()),
    ("features_json",pa.string()),
    ("description_json",pa.string()),
    ("categories_json",pa.string()),
    ("details_json",pa.string()),
    ("image_url",pa.string()),
])

#scrive righe in un file parquet, rows=listadiz, path=dove salvare, schema=schema PyArrow, writer=oggetto che gestisce la scrittura Parquet
def writerows_in_parquet(rows:list[dict],path:Path,schema:pa.schema,writer:Optional[pq.ParquetWriter]):
    if not rows:
        return writer

    df=pd.DataFrame(rows) #converte lista di dizionari in un dataframe pandas
    table=pa.Table.from_pandas(df,schema=schema,preserve_index=False) #converte dataframe in tabella pyarrow

    if writer is None:
        path.parent.mkdir(parents=True,exist_ok=True) #crea cartella di output
        writer=pq.ParquetWriter(path,schema=schema,compression="zstd") #crea un writer Parquet,compresso
    writer.write_table(table)
    return writer

In [8]:
#carichiamo metadati amazon reviews 2023 da Hugging Face per ogni categoria
#teniamo solo prodotti presenti anche nel QA e deduplichiamo parent_asin fra categorie
#creaiamo intes.parquet cioè tab prodotti finali

if ITEMS_OUT.exists(): #se items.parquet esiste già lo elimina
    ITEMS_OUT.unlink()

writer=None
buffer=[]
matched_items_total=0
global_items_seen=set()
items_id_by_category={}

for category in CATEGORIES: #cicla ogni categoria
    label=category["label"]
    hf_category=category["hf_category"]
    qa_asins=qa_info_by_category[label]["qa_asins"]

    meta_ds=load_dataset( #risultato salvato in meta_ds
        "McAuley-Lab/Amazon-Reviews-2023", #nome del dataset su HuggingFace
        f"raw_meta_{hf_category}", #metadati della categoria corrente
        split="full",
        trust_remote_code=True,
    )

    matched_items=0
    duplicates_skipped=0
    category_items=set()

    for row in tqdm(meta_ds,desc=f"Filtro metadati {label}"): #legge una riga prodotto alla  volta dal dataset huggingface
        parent_asin=row.get("parent_asin")
        if parent_asin not in qa_asins: #salta items non in qa storico
            continue
        if parent_asin in global_items_seen: #salta prodotto se compare in più categorie, senza duplicati
            duplicates_skipped+=1
            continue

        buffer.append({ #ggiunge prodotto al buffer
            "parent_asin":parent_asin,
            "source_category":label,
            "hf_category":hf_category,
            "qa_slug":category["qa_slug"],
            "category_group":category["group"],
            "title":row.get("title"),
            "category":get_category(row,label),
            "avg_rating":tofloat_or_none(row.get("average_rating")),
            "num_ratings":toint_or_none(row.get("rating_number")),
            "price":tofloat_or_none(row.get("price")),
            "store":row.get("store"),
            "features_json":tojson_string(row.get("features")),
            "description_json":tojson_string(row.get("description")),
            "categories_json":tojson_string(row.get("categories")),
            "details_json":tojson_string(row.get("details")),
            "image_url":image_url(row.get("images")),
        })
        matched_items+=1
        matched_items_total+=1
        category_items.add(parent_asin)
        global_items_seen.add(parent_asin)

        if len(buffer)>=50_000: #ogni 50k righe svuota il buffer e scrive nel parquet
            writer=writerows_in_parquet(buffer,ITEMS_OUT,ITEM_SCHEMA,writer)
            buffer=[]

    items_id_by_category[label]=category_items #salva insieme dei prodotti di quella categoria
    for row in category_summary_rows: #aggiorniamo riepilogo per categoria, con dati 
        if row["source_category"]==label:
            row["metadata_rows"] = int(len(meta_ds)) #prodotti metadati della categoria
            row["items_qa_metadata"] = int(matched_items) #prodotti salvati perchè sia in qa sia metadata
            row["duplicate_items_skipped"] = int(duplicates_skipped) #prodotti skipapti perchè gia presi in altre categorie
            break

    print(label,"| metadati:",len(meta_ds),"| prodotti QA ∩ metadata salvati:",matched_items,"| duplicati saltati:",duplicates_skipped)

writer=writerows_in_parquet(buffer,ITEMS_OUT,ITEM_SCHEMA,writer)#ultimo buffer rimasto
if writer is not None:
    writer.close()

items_df=pd.read_parquet(ITEMS_OUT)
items_id=set(items_df["parent_asin"])

print("Prodotti QA ∩ metadata 2023 totali:",len(items_id))
display(items_df.head())

C:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arsid\.cache\huggingface\hub\datasets--McAuley-Lab--Amazon-Reviews-2023. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Generating full split:   0%|          | 0/1610012 [00:00<?, ? examples/s]

Generating full split:   0%|          | 1110/1610012 [00:00<02:25, 11051.74 examples/s]

Generating full split:   0%|          | 2447/1610012 [00:00<02:09, 12404.30 examples/s]

Generating full split:   0%|          | 3815/1610012 [00:00<02:04, 12945.49 examples/s]

Generating full split:   0%|          | 5811/1610012 [00:00<02:05, 12831.66 examples/s]

... (5,917 outputs left out)

Filtro metadati Appliances: 100%|██████████| 94327/94327 [00:08<00:00, 11173.60it/s]

Appliances | metadati: 94327 | prodotti QA ∩ metadata salvati: 631 | duplicati saltati: 0


Prodotti QA ∩ metadata 2023 totali: 32267


,parent_asin,source_category,hf_category,qa_slug,category_group,title,category,avg_rating,num_ratings,price,store,features_json,description_json,categories_json,details_json,image_url
0,B00CFGZAT8,Electronics,Electronics,Electronics,tech,Dock Audio Extender Adapter Converter Cable fo...,Portable Audio & Accessories,3.6,45,NaN,iteck,[],[],"[""Electronics"", ""Computers & Accessories"", ""Ta...","""{\""Brand Name\"": \""iTeck\"", \""Item Weight\"": ...",https://m.media-amazon.com/images/I/41uTuqlHDS...
1,B00BGFTSGU,Electronics,Electronics,Electronics,tech,GGS Swivi HD DSLR LCD Universal Foldable Viewf...,Camera & Photo,3.7,52,149.00,Swivi,[],[],"[""Electronics"", ""Camera & Photo"", ""Accessories...","""{\""Product Dimensions\"": \""3.8 x 4.3 x 6.9 in...",https://m.media-amazon.com/images/I/41BUv5PjMD...
2,B00AZOIG9I,Electronics,Electronics,Electronics,tech,BM Premium 2-Pack of NP-85 Batteries and Charg...,Camera & Photo,4.7,345,22.99,BM Premium,"[""NP85 Li-ion Battery for FujiFilm FinePix SL2...","[""Bring your digital camera back to life with ...","[""Electronics"", ""Camera & Photo"", ""Accessories...","""{\""Package Dimensions\"": \""6.46 x 3.94 x 1.85...",https://m.media-amazon.com/images/I/71fbU-87Tx...
3,B00F361KLO,Electronics,Electronics,Electronics,tech,ZAGG InvisibleShield HD – EZ Apply Film Screen...,Computers,4.1,158,NaN,InvisibleShield,"[""100% Clear: Independent light transmission t...","[""Exceptionally clear, unbelievably thin, and ...","[""Electronics"", ""Computers & Accessories"", ""Ta...","""{\""Standing screen display size\"": \""9.7 Inch...",https://m.media-amazon.com/images/I/71XY4kKwcX...
4,B0029L7N76,Electronics,Electronics,Electronics,tech,Mobile Edge ECO Laptop Messenger (Eco-Friendly...,Computers,3.5,12,NaN,Mobile Edge,"[""Fits 17.3-Inch laptops. Laptop compartment d...","[""The 17.3-Inch ECO Messenger is the latest ad...","[""Electronics"", ""Computers & Accessories"", ""La...","""{\""Product Dimensions\"": \""4.25 x 18 x 13.5 i...",https://m.media-amazon.com/images/I/91JdX5IqgA...


In [9]:
# ora filtriamo QA e teniamo solo domande/risposte dei prodotti presenti anche nei metadati 2023
QA_SCHEMA=pa.schema([ #schema del file qa.parquet
    ("qa_id",pa.int64()),
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("question",pa.string()),
    ("answer",pa.string()),
    ("questionType",pa.string()),
    ("answerType",pa.string()),
    ("answerTime",pa.string()),
    ("unixTime",pa.int64()),
])

def iter_jsonl_gz(path:Path)->Iterable[dict]: #legge file .jsonl.gz
    with gzip.open(path,"rt",encoding="utf-8",errors="replace") as f:
        for line in f:
            line=line.strip() #pulisce stringa
            if line:
                yield json.loads(line)

if QA_OUT.exists(): #se qa.parquet esiste già lo elimina
    QA_OUT.unlink()

writer=None
buffer=[]
matched_qa_total=0

for category in CATEGORIES:
    label=category["label"]
    category_items=items_id_by_category[label] #insieme prodotti della categoria presa
    qa_strict_path=qa_info_by_category[label]["qa_strict_path"] #prende file qa convertito della categoria
    matched_qa=0

    for row in tqdm(iter_jsonl_gz(qa_strict_path),desc=f"Filtro QA {label}"): #legge riga per riga il file qa, con barra di avanzamento tqdm
        parent_asin=row.get("parent_asin")
        if parent_asin not in category_items: #skippa qa di prodotti non presenti in items.parquet
            continue

        buffer.append({ #aggiunge la riga qa al buffer
            "qa_id":int(row.get("qa_id")),
            "parent_asin":parent_asin,
            "source_category":label,
            "hf_category":category["hf_category"],
            "qa_slug":category["qa_slug"],
            "category_group":category["group"],
            "question":row.get("question"),
            "answer":row.get("answer"),
            "questionType":row.get("questionType"),
            "answerType":row.get("answerType"),
            "answerTime":row.get("answerTime"),
            "unixTime":toint_or_none(row.get("unixTime")),
        })
        matched_qa+=1
        matched_qa_total+=1

        if len(buffer)>=100_000: #quando buffer è pieno, scrive nel qa.parquet
            writer=writerows_in_parquet(buffer,QA_OUT,QA_SCHEMA,writer)
            buffer=[]

    for row_summary in category_summary_rows:
        if row_summary["source_category"]==label:
            row_summary["qa_rows_after_metadata"] = int(matched_qa)
            row_summary["qa_products_after_metadata"] = int(len(category_items))
            break
    print(label,"| QA filtrate:",matched_qa)

writer=writerows_in_parquet(buffer,QA_OUT,QA_SCHEMA,writer)
if writer is not None:
    writer.close()

qa_df=pd.read_parquet(QA_OUT)
print("QA filtrate totali:",len(qa_df))
display(qa_df.head())

Filtro QA Electronics: 0it [00:00, ?it/s]

Filtro QA Electronics: 18612it [00:00, 186115.74it/s]

Filtro QA Electronics: 37677it [00:00, 188763.44it/s]

Filtro QA Electronics: 57441it [00:00, 192813.56it/s]

Filtro QA Electronics: 76723it [00:00, 177155.31it/s]

Filtro QA Electronics: 96097it [00:00, 182813.72it/s]

... (31 outputs left out)

Filtro QA Appliances: 9011it [00:00, 200873.08it/s]

Appliances | QA filtrate: 4862


QA filtrate totali: 256998


,qa_id,parent_asin,source_category,hf_category,qa_slug,category_group,question,answer,questionType,answerType,answerTime,unixTime
0,0,0594033926,Electronics,Electronics,Electronics,tech,Is this cover the one that fits the old nook c...,Yes this fits both the nook color and the same...,yes/no,Y,"Dec 27, 2013",1.388131e+09
1,1,0594033926,Electronics,Electronics,Electronics,tech,Does it fit Nook GlowLight?,No. The nook color or color tablet,yes/no,N,"Jan 5, 2015",1.420445e+09
2,2,0594033926,Electronics,Electronics,Electronics,tech,Would it fit Nook 1st Edition? 4.9in x 7.7in ?,I don't think so. The nook color is 5 x 8 so n...,open-ended,NaN,2 days ago,NaN
3,3,0594033926,Electronics,Electronics,Electronics,tech,Will this fit a Nook Color that's 5 x 8?,yes,yes/no,Y,17 days ago,NaN
4,4,0594033926,Electronics,Electronics,Electronics,tech,will this fit the Samsung Galaxy Tab 4 Nook 10.1,"No, the tab is smaller than the 'color'",yes/no,N,"Feb 10, 2015",1.423555e+09


In [10]:
#scansioniamo reviews 2023 per tutte le categorie e salviamo solo interazioni dei prodotti che abbiamo già
#se ratings.parquet esiste già e FORCE_REBUILD_RATINGS è False, non lo ricreiamo
#creiamo quindi ratings.parquet
rating_fields=[ #lista di colonne per file di rating
    ("user_id",pa.string()),
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("asin",pa.string()),
    ("rating",pa.float64()),
    ("timestamp",pa.int64()),
    ("verified_purchase",pa.bool_()),
]
if SAVE_TEXT_OFREVIEW: #se piu avanti vogliamo usare recensioni per sistema qa o per altri sistemi recomendation
    rating_fields.extend([
        ("review_title",pa.string()),
        ("review_text",pa.string()),
    ])
RATING_SCHEMA=pa.schema(rating_fields)

if RATINGS_OUT.is_file() and RATINGS_OUT.stat().st_size>0 and not FORCE_REBUILD_RATINGS: #se ratingsout esiste, non è vuoto e forcerebuild è false
    #Blocca il riuso di rating appartenenti a una configurazione diversa
    cached_summary=json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    expected_categories=[category["label"] for category in CATEGORIES]
    if cached_summary.get("dataset_name")!=DATASET_NAME or cached_summary.get("dataset_label")!=DATASET_LABEL or cached_summary.get("categories")!=expected_categories:
        raise ValueError("Provenienza dei rating incoerente: ricostruire gli output Amazon-like")
    cached_ratings=pd.read_parquet(RATINGS_OUT,columns=["parent_asin","source_category","hf_category","qa_slug","category_group"])
    if set(cached_ratings["source_category"])!=set(expected_categories) or len(cached_ratings)!=cached_summary.get("ratings"):
        raise ValueError("Categorie o conteggi dei rating incoerenti con summary.json")
    for category in CATEGORIES:
        category_ratings=cached_ratings[cached_ratings["source_category"]==category["label"]]
        if not set(category_ratings["parent_asin"]).issubset(items_id_by_category[category["label"]]):
            raise ValueError("Prodotti dei rating assenti dai metadati della categoria")
        for column,key in [("hf_category","hf_category"),("qa_slug","qa_slug"),("category_group","group")]:
            if not category_ratings[column].eq(category[key]).all():
                raise ValueError(f"Provenienza dei rating incoerente: {column}")
    if not set(RATING_SCHEMA.names).issubset(pq.ParquetFile(RATINGS_OUT).schema_arrow.names):
        raise ValueError("Schema dei rating incompatibile con la configurazione")
    del cached_ratings,category_ratings
    n_rows=pq.ParquetFile(RATINGS_OUT).metadata.num_rows
    print(f"ratings.parquet già presente: {RATINGS_OUT}")
    print(f"Righe già presenti: {n_rows:,}")
    print("Saltiamo il filtraggio dei rating.")
else: #sennò elimina quello vecchio se presente, e lo ricreiamo
    if RATINGS_OUT.exists():
        RATINGS_OUT.unlink()

    writer=None
    buffer=[]
    matched_ratings_total=0
    scanned_total=0

    for category in CATEGORIES: #cicla categorie
        label=category["label"]
        hf_category=category["hf_category"]
        category_items=items_id_by_category[label]
        max_scan=None
        if isinstance(MAX_REVIEWS_SCAN_BY_CATEGORY,dict): #dipende se max reviews è dizionario prende limite per categoria
            max_scan=MAX_REVIEWS_SCAN_BY_CATEGORY.get(label)
        elif MAX_REVIEWS_SCAN_BY_CATEGORY is not None: #se è limite per tutte uguale
            max_scan=MAX_REVIEWS_SCAN_BY_CATEGORY
        #scansioniamo tutte le reviews
        review_stream=load_dataset( #carichi review della categoria corrente
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_review_{hf_category}",
            split="full",
            streaming=True, #legge righe progressivamente
            trust_remote_code=True,
        )

        matched_ratings=0
        scanned=0

        for row in tqdm(review_stream,desc=f"Filtro rating {label}"): #legge una review alla volta
            if max_scan is not None and scanned>=max_scan: #se raggiunto limite, interrompe
                break
            scanned+=1
            parent_asin=row.get("parent_asin")
            if parent_asin not in category_items: #skippa reviews di prodotti non nel catalogo finale
                continue

            out_row={ #riga rating finale
                "user_id":row.get("user_id"),
                "parent_asin":parent_asin,
                "source_category":label,
                "hf_category":hf_category,
                "qa_slug":category["qa_slug"],
                "category_group":category["group"],
                "asin":row.get("asin"),
                "rating":tofloat_or_none(row.get("rating")),
                "timestamp":toint_or_none(row.get("timestamp")),
                "verified_purchase":row.get("verified_purchase"),
            }
            if SAVE_TEXT_OFREVIEW: #piu testo se lo vogliamo, per ora no, per alleggerire e testare modello base
                out_row["review_title"]=row.get("title")
                out_row["review_text"]=row.get("text")

            buffer.append(out_row)
            matched_ratings+=1
            matched_ratings_total+=1

            if len(buffer)>=200_000: #buffer pieno,scrivo su ratings.parquet
                writer=writerows_in_parquet(buffer,RATINGS_OUT,RATING_SCHEMA,writer)
                buffer=[]

        scanned_total+=scanned
        for row_summary in category_summary_rows:
            if row_summary["source_category"]==label:
                row_summary["reviews_scanned"] = int(scanned)
                row_summary["ratings_saved"] = int(matched_ratings)
                break
        print(label,"| review scansionate:",scanned,"| rating salvati:",matched_ratings)

    writer=writerows_in_parquet(buffer,RATINGS_OUT,RATING_SCHEMA,writer)
    if writer is not None:
        writer.close()

    print("Review scansionate totali:",scanned_total)
    print("Rating salvati totali:",matched_ratings_total)
    print("Output rating:",RATINGS_OUT)

ratings.parquet già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection\ratings.parquet
Righe già presenti: 5,245,310
Saltiamo il filtraggio dei rating.


In [11]:
#allineamento items e qa ai soli prodotti che hanno rating
#così il dataset finale contiene prodotti con metadata + QA + rating
ratings_df=pd.read_parquet(RATINGS_OUT,columns=["parent_asin","source_category"]) #carica solo colonne utili
rating_products=set(ratings_df["parent_asin"]) #insieme di prodotti che hanno almeno un rating

items_df=items_df[ #filtra items df tenendo solo quelli presenti in ratings products
    items_df["parent_asin"].isin(rating_products)
].copy()
items_df.to_parquet(ITEMS_OUT,index=False)
items_id=set(items_df["parent_asin"])

valid_products=set(items_df["parent_asin"]) #filtra anche qadf
qa_df=qa_df[
    qa_df["parent_asin"].isin(valid_products)
].copy()
qa_df.to_parquet(QA_OUT,index=False)

category_final_counts=( #conta prodotti finali per categoria
    items_df.groupby("source_category")["parent_asin"]
    .nunique()
    .rename("final_products_qa_metadata_rating")
    .reset_index()
)

for row_summary in category_summary_rows: #aggiorna riepilogo per categoria, aggiungendo num prodotti finali
    label=row_summary["source_category"]
    final_count=category_final_counts.loc[
        category_final_counts["source_category"]==label,
        "final_products_qa_metadata_rating"
    ]
    row_summary["final_products_qa_metadata_rating"]=int(final_count.iloc[0]) if len(final_count) else 0

category_summary_df=pd.DataFrame(category_summary_rows)
category_summary_df.to_csv(CATEGORY_SUMMARY_PATH,index=False)

print("Prodotti finali con metadata + QA + rating:",len(items_df))
print("Prodotti con QA dopo aver filtrato:",qa_df["parent_asin"].nunique())
print("Righe QA dopo filtro:",len(qa_df))
print("Riepilogo per categoria:",CATEGORY_SUMMARY_PATH)
display(category_summary_df)

Prodotti finali con metadata + QA + rating: 32266
Prodotti con QA dopo aver filtrato: 32266
Righe QA dopo filtro: 256988
Riepilogo per categoria: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection\category_summary.csv


,source_category,hf_category,qa_slug,category_group,qa_rows,qa_products,metadata_rows,items_qa_metadata,duplicate_items_skipped,qa_rows_after_metadata,qa_products_after_metadata,final_products_qa_metadata_rating
0,Electronics,Electronics,Electronics,tech,314263,39371,1610012,22975,0,180986,22975,22974
1,Cell Phones and Accessories,Cell_Phones_and_Accessories,Cell_Phones_and_Accessories,tech,85865,10407,1288490,4705,0,38582,4705,4705
2,Video Games,Video_Games,Video_Games,tech,13307,1188,137269,607,0,6601,607,607
3,Software,Software,Software,tech,10636,1191,89251,647,0,5723,647,647
4,Office Products,Office_Products,Office_Products,office,43608,5681,710503,2702,0,20244,2702,2702
5,Appliances,Appliances,Appliances,home,9011,1153,94327,631,0,4862,631,631


In [12]:
def parquet_count_rows(path:Path)->int: #funzione che conta righe di un file parquet
    return pq.ParquetFile(path).metadata.num_rows

summary={ #dizionario riassuntivo
    "dataset_name":DATASET_NAME,
    "dataset_label":DATASET_LABEL,
    "categories":[category["label"] for category in CATEGORIES],
    "items":parquet_count_rows(ITEMS_OUT),
    "qa":parquet_count_rows(QA_OUT),
    "ratings":parquet_count_rows(RATINGS_OUT),
    "category_summary_path":str(CATEGORY_SUMMARY_PATH),
}

SUMMARY_PATH.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
summary

{'dataset_name': 'test_dataset2',
 'dataset_label': 'AmazonLike_v1',
 'categories': ['Electronics',
  'Cell Phones and Accessories',
  'Video Games',
  'Software',
  'Office Products',
  'Appliances'],
 'items': 32266,
 'qa': 256988,
 'ratings': 5245310,
 'category_summary_path': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\data\\processed\\test_dataset2_intersection\\category_summary.csv'}